In [39]:
# 导入必要的库
import os
# 导入我们刚刚修改的脚本中的核心函数
# 假设 create_code_summary.py 与你的 notebook 在同一目录下
# 如果不在同一目录，你可能需要先将该目录添加到 sys.path
# import sys
# sys.path.append('/path/to/directory/containing/the/script')
from create_code_summary import generate_code_summary

# --- 配置 ---

# 1. 指定你要扫描代码的目录路径
#    将其替换为你的实际路径
#    Windows 示例: code_directory = r"C:\Users\YourName\Projects\MyProject"
#    Linux/macOS 示例: code_directory = "/home/yourname/projects/myproject"
code_directory = "step_3" # <--- 修改这里！

# 2. (可选) 指定输出文件的名字
output_file_name = "code_meta_resample_3.txt"

# --- 执行调用 ---

print(f"将要扫描目录: {code_directory}")
print(f"查找的文件类型: .py, .sh")
print(f"输出文件名: {output_file_name}")
print("-" * 30)

# 调用导入的函数，传入目录路径和可选的输出文件名
# 函数会执行查找和写入操作，并打印过程信息
output_path = generate_code_summary(code_directory, output_filename=output_file_name)

print("-" * 30)
if output_path:
    print(f"操作成功完成！结果文件保存在: {output_path}")
else:
    print("操作未生成输出文件（可能未找到文件或发生错误）。")

将要扫描目录: step_3
查找的文件类型: .py, .sh
输出文件名: code_meta_resample_3.txt
------------------------------
开始在 'step_3' 及其子目录中搜索扩展名为 ('.py', '.sh') 的文件...
搜索完成，共找到 5 个符合条件的文件。
开始将代码写入到 'code_meta_resample_3.txt'...
写入完成。成功写入 5 个文件，读取/写入失败 0 个文件。
结果已保存到：/project/hrao/Meta-Resample/code_meta_resample_3.txt
------------------------------
操作成功完成！结果文件保存在: /project/hrao/Meta-Resample/code_meta_resample_3.txt


In [ ]:
Meta_Aug_NER/augmentation

In [1]:
import re
import ast
import matplotlib.pyplot as plt
import os
from collections import defaultdict
import numpy as np # Needed to potentially eval numpy types if ast fails

# --- Configuration ---
LOG_FILE_PATH = "log/meta_aug/mit_movie_trivia_meta_aug_run1_11611.log"
OUTPUT_DIR = "training_plots"
METRICS_TO_PLOT = ['overall_f1', 'overall_precision', 'overall_recall', 'overall_accuracy']
LOSSES_TO_PLOT = ['Inner Loss', 'Outer Loss', 'KL Loss', 'Warmup Avg Loss']
# --- End Configuration ---

def parse_log_file(log_path):
    """
    Parses the log file to extract steps and metric/loss values.
    """
    # Using defaultdict to easily append steps and values
    data = defaultdict(lambda: {'steps': [], 'values': []})

    # Regex to find evaluation metric lines (Warmup or Meta)
    # Captures step number and the dictionary string
    metrics_pattern = re.compile(r"INFO - __main__ - (?:Warmup|Meta) Step (\d+) Meta-Validation Metrics: (\{.*\})")

    # Regex for Meta Step loss lines
    meta_loss_pattern = re.compile(
        r"INFO - __main__ - Meta Step (\d+) \| Inner Loss: ([\d.]+) \|(?: Outer Loss: ([\d.]+|inf) \|)?(?: KL Loss: ([\d.]+) \|)?.*"
    )
    # Regex for Warmup Step loss lines
    warmup_loss_pattern = re.compile(
        r"INFO - __main__ - Warmup Step (\d+), Avg Loss: ([\d.]+)"
    )


    print(f"Parsing log file: {log_path}")
    try:
        with open(log_path, 'r', encoding='utf-8') as f:
            for line in f:
                # --- Parse Evaluation Metrics ---
                match = metrics_pattern.search(line)
                if match:
                    step = int(match.group(1))
                    metrics_str = match.group(2)

                    try:
                        # Preprocess the string to remove numpy types for ast.literal_eval
                        processed_str = metrics_str.replace('np.float64(', '').replace('np.int64(', '').replace(')', '')
                        # Replace potential inf/-inf that ast cannot handle
                        processed_str = processed_str.replace('inf', 'float("inf")').replace('-inf', 'float("-inf")')

                        # Safely evaluate the string as a Python literal
                        metrics_dict = ast.literal_eval(processed_str)

                        for metric_name in METRICS_TO_PLOT:
                            if metric_name in metrics_dict:
                                data[metric_name]['steps'].append(step)
                                data[metric_name]['values'].append(metrics_dict[metric_name])
                            #else:
                            #    print(f"Warning: Metric '{metric_name}' not found in step {step} metrics dict.")

                    except (SyntaxError, ValueError) as e:
                        print(f"Warning: Could not parse metrics dictionary at step {step}. Error: {e}")
                        print(f"Problematic string: {metrics_str[:200]}...") # Print part of the string
                    continue # Move to next line once metrics are parsed

                # --- Parse Meta Step Losses ---
                match = meta_loss_pattern.search(line)
                if match:
                    step = int(match.group(1))
                    inner_loss = float(match.group(2))
                    outer_loss_str = match.group(3)
                    kl_loss_str = match.group(4)

                    data['Inner Loss']['steps'].append(step)
                    data['Inner Loss']['values'].append(inner_loss)

                    if outer_loss_str and outer_loss_str != 'inf':
                        data['Outer Loss']['steps'].append(step)
                        data['Outer Loss']['values'].append(float(outer_loss_str))

                    if kl_loss_str:
                         data['KL Loss']['steps'].append(step)
                         data['KL Loss']['values'].append(float(kl_loss_str))
                    continue

                # --- Parse Warmup Step Losses ---
                match = warmup_loss_pattern.search(line)
                if match:
                    step = int(match.group(1))
                    avg_loss = float(match.group(2))
                    data['Warmup Avg Loss']['steps'].append(step)
                    data['Warmup Avg Loss']['values'].append(avg_loss)
                    continue


    except FileNotFoundError:
        print(f"Error: Log file not found at {log_path}")
        return None
    except Exception as e:
        print(f"An error occurred during parsing: {e}")
        return None

    # Sort data by step for plotting
    for key in data:
        steps = data[key]['steps']
        values = data[key]['values']
        sorted_pairs = sorted(zip(steps, values))
        if sorted_pairs:
             data[key]['steps'], data[key]['values'] = zip(*sorted_pairs)
        else:
             data[key]['steps'], data[key]['values'] = [], []


    print("Parsing complete.")
    # Print found keys and number of data points for verification
    for key, val in data.items():
        print(f"  Found {len(val['steps'])} data points for '{key}'")

    return data

def plot_metrics(data, output_dir):
    """
    Plots the parsed metrics and saves the figures.
    """
    if not data:
        print("No data parsed, skipping plotting.")
        return

    os.makedirs(output_dir, exist_ok=True)
    print(f"Saving plots to: {output_dir}")

    # --- Plot F1 Score and other main metrics ---
    plt.figure(figsize=(12, 8))

    metrics_plotted = 0
    for metric_name in METRICS_TO_PLOT:
        if metric_name in data and data[metric_name]['steps']:
            plt.plot(data[metric_name]['steps'], data[metric_name]['values'], marker='.', linestyle='-', label=metric_name)
            metrics_plotted += 1
        else:
            print(f"Skipping plot for '{metric_name}' - No data found.")

    if metrics_plotted > 0:
        plt.xlabel("Global Step")
        plt.ylabel("Score")
        plt.title("Evaluation Metrics during Training")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plot_filename = os.path.join(output_dir, "evaluation_metrics_plot.png")
        plt.savefig(plot_filename)
        print(f"Saved metrics plot to {plot_filename}")
        plt.close() # Close the figure to free memory
    else:
        print("No evaluation metrics found to plot.")


    # --- Plot Losses ---
    plt.figure(figsize=(12, 8))

    losses_plotted = 0
    for loss_name in LOSSES_TO_PLOT:
         if loss_name in data and data[loss_name]['steps']:
            plt.plot(data[loss_name]['steps'], data[loss_name]['values'], marker='.', linestyle='-', label=loss_name)
            losses_plotted += 1
         else:
            print(f"Skipping plot for '{loss_name}' - No data found.")

    if losses_plotted > 0:
        plt.xlabel("Global Step")
        plt.ylabel("Loss Value")
        plt.title("Training Losses during Training")
        # Optional: Use log scale if losses vary greatly
        # plt.yscale('log')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plot_filename = os.path.join(output_dir, "loss_plot.png")
        plt.savefig(plot_filename)
        print(f"Saved loss plot to {plot_filename}")
        plt.close()
    else:
         print("No loss values found to plot.")


if __name__ == "__main__":
    parsed_data = parse_log_file(LOG_FILE_PATH)
    if parsed_data:
        plot_metrics(parsed_data, OUTPUT_DIR)
    print("Plotting script finished.")

Parsing log file: log/meta_aug/mit_movie_trivia_meta_aug_run1_11611.log
Parsing complete.
  Found 34 data points for 'Warmup Avg Loss'
  Found 12 data points for 'overall_f1'
  Found 12 data points for 'overall_precision'
  Found 12 data points for 'overall_recall'
  Found 12 data points for 'overall_accuracy'
  Found 86 data points for 'Inner Loss'
  Found 86 data points for 'Outer Loss'
  Found 86 data points for 'KL Loss'
Saving plots to: training_plots
Saved metrics plot to training_plots/evaluation_metrics_plot.png
Saved loss plot to training_plots/loss_plot.png
Plotting script finished.


In [4]:
import re
import ast

def extract_epoch_data(log_file_path):
    """
    Extracts specified parameters for each meta-training epoch from the log file,
    handling the multi-line, space-separated Rho format using state-based parsing.

    Args:
        log_file_path (str): The path to the log file.

    Returns:
        dict: A dictionary where keys are epoch numbers and values are
              dictionaries containing 'losses', 'policy', and 'rho' data.
              Returns None if the file cannot be opened.
    """
    epoch_data = {}
    current_epoch = None
    reading_rho = False
    rho_lines_buffer = [] # To accumulate lines belonging to Rho list

    # Regex patterns
    loss_pattern = re.compile(r'Meta Epoch (\d+) Average Losses: Inner=([\d.]+), Outer=([\d.]+), KL=([\d.]+)', re.IGNORECASE)
    policy_pattern = re.compile(r'Meta Epoch (\d+) Policy State: P_Global=([\d.]+), Alpha=([\d.]+), Op Probs=(\[.*?\])', re.IGNORECASE)
    # Rho pattern just to find the *start* of the line
    rho_start_pattern = re.compile(r'Meta Epoch (\d+) Rho\s*:', re.IGNORECASE)

    try:
        with open(log_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                # --- State: Reading Rho lines ---
                if reading_rho:
                    # Append the relevant part of the line to the buffer
                    # We assume rho data continues immediately on the next lines
                    rho_lines_buffer.append(line.strip())
                    # Check if this line contains the closing bracket
                    if ']' in line:
                        reading_rho = False # Done reading rho for this epoch
                        # Process the accumulated buffer
                        full_rho_str = " ".join(rho_lines_buffer)
                        # Extract the part actually within the brackets robustly
                        start_bracket_index = full_rho_str.find('[')
                        end_bracket_index = full_rho_str.rfind(']') # Use rfind for the last bracket

                        if start_bracket_index != -1 and end_bracket_index != -1 and end_bracket_index > start_bracket_index:
                            rho_content_str = full_rho_str[start_bracket_index + 1 : end_bracket_index]
                            try:
                                # Parse the space-separated floats
                                rho_values = [float(x) for x in rho_content_str.split() if x] # Split handles spaces
                                if current_epoch is not None and current_epoch in epoch_data:
                                    epoch_data[current_epoch]['rho'] = rho_values
                                else:
                                     print(f"Warning: Processed Rho data but current_epoch ({current_epoch}) was invalid or missing.")
                            except ValueError as e:
                                print(f"Warning: Could not parse Rho floats for epoch {current_epoch}. Storing raw buffer. Error: {e}")
                                if current_epoch is not None and current_epoch in epoch_data:
                                    epoch_data[current_epoch]['rho'] = full_rho_str # Fallback: store raw accumulated string
                        else:
                            print(f"Warning: Could not find valid brackets '[]' in accumulated Rho buffer for epoch {current_epoch}")
                            if current_epoch is not None and current_epoch in epoch_data:
                                epoch_data[current_epoch]['rho'] = full_rho_str # Fallback

                        rho_lines_buffer = [] # Clear buffer for next time
                        # Reset current_epoch only after successfully processing the last part (Rho)
                        current_epoch = None
                    # Continue to next line whether or not the end bracket was found on *this* line
                    continue

                # --- State: Not reading Rho ---

                # Check for Average Losses line
                match = loss_pattern.search(line)
                if match:
                    # If we were unexpectedly reading Rho, process what we have (error state)
                    if reading_rho:
                         print(f"Warning: Found new Epoch Loss line while still reading Rho for epoch {current_epoch}. Discarding previous Rho buffer.")
                         reading_rho = False
                         rho_lines_buffer = []
                    # Process the loss line
                    current_epoch = int(match.group(1))
                    if current_epoch not in epoch_data:
                        epoch_data[current_epoch] = {}
                    epoch_data[current_epoch]['losses'] = {
                        'Inner': float(match.group(2)),
                        'Outer': float(match.group(3)),
                        'KL': float(match.group(4))
                    }
                    # Don't reset current_epoch yet, wait for Policy and Rho lines
                    continue

                # Check for Policy State line
                match = policy_pattern.search(line)
                if match:
                    epoch = int(match.group(1))
                    # Ensure it matches the current epoch being processed
                    if epoch == current_epoch and current_epoch is not None:
                        try:
                            op_probs_list = ast.literal_eval(match.group(4))
                        except (ValueError, SyntaxError):
                            op_probs_list = match.group(4) # Keep as string if eval fails

                        if current_epoch in epoch_data: # Check if epoch exists from loss line
                            epoch_data[current_epoch]['policy'] = {
                                'P_Global': float(match.group(2)),
                                'Alpha': float(match.group(3)),
                                'Op Probs': op_probs_list
                            }
                        else:
                             print(f"Warning: Found Policy state for epoch {epoch} but no preceding Loss data found.")
                    # Don't reset current_epoch yet, wait for Rho
                    continue

                # Check for the start of the Rho line
                match = rho_start_pattern.search(line)
                if match:
                    epoch = int(match.group(1))
                    if epoch == current_epoch and current_epoch is not None:
                        # Start reading Rho data
                        reading_rho = True
                        # Extract the part after ':' on the *first* line
                        start_content_index = line.find(':') + 1
                        first_rho_part = line[start_content_index:].strip()
                        rho_lines_buffer = [first_rho_part] # Initialize buffer

                        # Handle edge case: Rho might be on a single line
                        if '[' in first_rho_part and ']' in first_rho_part:
                            reading_rho = False # Stop reading immediately
                            full_rho_str = " ".join(rho_lines_buffer) # Should just be the first part
                            start_bracket_index = full_rho_str.find('[')
                            end_bracket_index = full_rho_str.rfind(']')

                            if start_bracket_index != -1 and end_bracket_index != -1 and end_bracket_index > start_bracket_index:
                                rho_content_str = full_rho_str[start_bracket_index + 1 : end_bracket_index]
                                try:
                                    rho_values = [float(x) for x in rho_content_str.split() if x]
                                    if current_epoch in epoch_data:
                                        epoch_data[current_epoch]['rho'] = rho_values
                                    else:
                                        print(f"Warning: Found single-line Rho but epoch {current_epoch} not in data.")
                                except ValueError as e:
                                    print(f"Warning: Could not parse single-line Rho floats for epoch {current_epoch}. Error: {e}")
                                    if current_epoch in epoch_data:
                                        epoch_data[current_epoch]['rho'] = full_rho_str
                            else:
                                 print(f"Warning: Could not find valid brackets '[]' in single-line Rho for epoch {current_epoch}")
                                 if current_epoch in epoch_data:
                                     epoch_data[current_epoch]['rho'] = full_rho_str

                            rho_lines_buffer = [] # Clear buffer
                            current_epoch = None # Reset epoch after processing Rho

                    # No continue here, let loop proceed normally if not single line


    except FileNotFoundError:
        print(f"Error: Log file not found at {log_file_path}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        # raise e # Uncomment to see the full traceback
        return None

    # Final check in case the log ended while reading Rho
    if reading_rho:
        print(f"Warning: Log file ended while reading Rho data for epoch {current_epoch}. Discarding buffer.")


    return epoch_data

def print_extracted_data(epoch_data):
    """Prints the extracted epoch data in a formatted way."""
    if not epoch_data:
        print("No epoch data found or error occurred during extraction.")
        return

    # Sort epochs for ordered printing
    sorted_epochs = sorted(epoch_data.keys())

    for epoch in sorted_epochs:
        print(f"========== Meta Epoch {epoch} ==========")
        data = epoch_data[epoch]

        if 'losses' in data:
            losses = data['losses']
            print(f"  Average Losses: Inner={losses.get('Inner', 'N/A'):.4f}, Outer={losses.get('Outer', 'N/A'):.4f}, KL={losses.get('KL', 'N/A'):.4f}")
        else:
            print("  Average Losses: Not found")

        if 'policy' in data:
            policy = data['policy']
            # Format Op Probs for cleaner printing if it's a list
            op_probs = policy.get('Op Probs', 'N/A')
            if isinstance(op_probs, list):
                 # Ensure floats are formatted, handle potential strings if eval failed
                 op_probs_str = '[' + ', '.join(f"'{val}'" if isinstance(val, str) else f"{float(val):.3f}" for val in op_probs) + ']'
            else:
                 op_probs_str = str(op_probs) # Keep as string if not list

            print(f"  Policy State: P_Global={policy.get('P_Global', 'N/A'):.3f}, Alpha={policy.get('Alpha', 'N/A'):.3f}, Op Probs={op_probs_str}")
        else:
            print("  Policy State: Not found")

        if 'rho' in data:
             rho = data['rho']
             # Format Rho for cleaner printing if it's a list
             if isinstance(rho, list):
                 # Use space separation for Rho as in the original log
                 rho_str = "[" + " ".join(f"{val:.7f}" for val in rho) + "]"
             else:
                 # If it's the raw buffer string (fallback), print it directly
                 rho_str = str(rho)
             print(f"  Rho: {rho_str}") # Indented for consistency
        else:
            # If Rho wasn't found even after multi-line logic
            print("  Rho: Not found")

        print("-" * 30)

# --- Main Execution ---
# Use the correct path to your log file
log_file = "log/meta_aug/mit_movie_trivia_meta_aug_run1_11642.log" # Make sure this path is correct
extracted_data = extract_epoch_data(log_file)
print_extracted_data(extracted_data)

========== Meta Epoch 1 ==========
  Average Losses: Inner=2.3401, Outer=2.1894, KL=0.1176
  Policy State: P_Global=0.100, Alpha=2.126, Op Probs=['0.333', '0.334', '0.333']
  Rho: [1.9982469 2.0016716 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 2.0016716 1.9982469 2.0016716 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 1.9982469 2.0016716 2.0016716 1.9982469]
------------------------------
========== Meta Epoch 2 ==========
  Average Losses: Inner=0.8549, Outer=1.1885, KL=0.1169
  Policy State: P_Global=0.100, Alpha=2.125, Op Probs=['0.333', '0.335', '0.333']
  Rho: [1.9964938 2.0033430 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 2.0033430 1.9964938 2.0033430 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 1.9964938 2.0033430 2.0033430 1.9964938]
------------------------------
========== Meta Epoch 3 ==========
  Average Losses: Inner=0.5346, Outer=0